# ZeroHunter — تدريب النموذج وتصدير GGUF

تأكد من اختيار **Runtime → Change runtime type → T4 GPU** قبل تشغيل الخلايا.

In [ ]:
%cd /content
!apt-get update -qq && apt-get install -y -qq cmake libcurl4-openssl-dev build-essential > /dev/null
!pip install -q --no-cache-dir unsloth gguf protobuf sentencepiece
!rm -rf zh_repo && git clone --depth 1 https://github.com/sdxdxa56-design/jdjsjsh.git zh_repo
%cd /content/zh_repo

In [ ]:
# تحديث بيانات الإنترنت قبل البدء وتدريب النموذج مع دمج ملفات التجربة
!python3 -c "from core.knowledge import build_curated_dataset; build_curated_dataset()" || true
!python3 train.py

In [ ]:
import os
import subprocess
from getpass import getpass

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
except Exception:
    pass

if not os.environ.get("HF_TOKEN"):
    try:
        os.environ["HF_TOKEN"] = getpass("أدخل HF_TOKEN لرفع النموذج (أو اضغط Enter للتخطي نحو الخادم السريع): ").strip()
    except Exception:
        os.environ["HF_TOKEN"] = ""

if os.environ.get("HF_TOKEN"):
    subprocess.run(["python3", "upload_hf.py"], check=False)
else:
    print("تم تخطي الرفع إلى Hugging Face مؤقتاً، وجاري الانتقال لتشغيل الخادم السريع.")

### خلية 4 (تشغيل فوري في 30 ثانية بدون تدريب): خادم الدردشة السريع (1–3 ثوانٍ للرد)

لتشغيل الدردشة السريعة فوراً: شغّل هذه الخلية وحدها مباشرة (لا تضغط Run all لأن الخلية 2 تبدأ تدريباً يستغرق 25–40 دقيقة).

In [ ]:
import os
import re
import time
import urllib.request
import subprocess
from pathlib import Path

if not Path("web_app.py").exists():
    os.chdir("/content")
    if not Path("/content/zh_repo/web_app.py").exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/sdxdxa56-design/jdjsjsh.git", "/content/zh_repo"], check=True)
    os.chdir("/content/zh_repo")

subprocess.run(["pip", "install", "-q", "flask", "requests", "huggingface_hub"], check=False)
wheel_res = subprocess.run(
    ["pip", "install", "-q", "--only-binary=:all:", "--extra-index-url", "https://abetlen.github.io/llama-cpp-python/whl/cu124", "llama-cpp-python"],
    check=False,
)
if wheel_res.returncode != 0:
    subprocess.run(["pip", "install", "-q", "--prefer-binary", "llama-cpp-python"], check=False)

if not Path("/usr/local/bin/cloudflared").exists():
    subprocess.run(["wget", "-q", "-O", "/usr/local/bin/cloudflared", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
    subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"], check=True)

from huggingface_hub import hf_hub_download

gguf_candidates = sorted(Path(".").rglob("*.gguf"), key=lambda p: p.stat().st_size, reverse=True)
if gguf_candidates:
    model_path = str(gguf_candidates[0].resolve())
else:
    model_path = hf_hub_download(
        repo_id="Qwen/Qwen2.5-Coder-3B-Instruct-GGUF",
        filename="qwen2.5-coder-3b-instruct-q4_k_m.gguf",
    )

env = os.environ.copy()
env["ZEROHUNTER_LOCAL_MODEL"] = model_path
with open("web_app.log", "w") as web_out:
    subprocess.Popen(["python3", "web_app.py"], env=env, stdout=web_out, stderr=subprocess.STDOUT)

server_ready = False
for _ in range(30):
    try:
        with urllib.request.urlopen("http://127.0.0.1:5000/", timeout=2) as resp:
            if resp.status < 500:
                server_ready = True
                break
    except Exception:
        time.sleep(1)

with open("cf.log", "w") as cf_out:
    subprocess.Popen(
        ["/usr/local/bin/cloudflared", "tunnel", "--url", "http://127.0.0.1:5000", "--no-autoupdate"],
        stdout=cf_out,
        stderr=subprocess.STDOUT,
    )

public_url = ""
for _ in range(45):
    time.sleep(1)
    if Path("cf.log").exists():
        log_txt = Path("cf.log").read_text(encoding="utf-8", errors="ignore")
        match = re.search(r"https://[A-Za-z0-9-]+\.trycloudflare\.com", log_txt)
        if match:
            public_url = match.group(0)
            break

if public_url and server_ready:
    print("خادم ZeroHunter السريع يعمل الآن!")
    print("انسخ هذا الرابط وضعه في Secrets باسم ZEROHUNTER_LOCAL_URL:")
    print(public_url)
else:
    print("تعذر إتمام التشغيل السريع:", {"server_ready": server_ready, "public_url": public_url})
    if Path("web_app.log").exists():
        print(Path("web_app.log").read_text(encoding="utf-8", errors="ignore")[-1000:])